# Adaptive pre-training
- TSDAE method for unsupervised training on Target Domain
- SBERT method for supervised training on Non-Target Domain 

## 1- TSDAE 

In [ ]:
import csv, glob, gzip, random
import xml.etree.ElementTree as ET
import nltk
nltk.download('punkt', quiet=True); nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import sent_tokenize

MAX_PAIRS, MAX_SENTENCES, MIN_ABS_WORDS = 200_000, 300_000, 40
random.seed(42)

In [81]:
from google.colab import drive
drive.mount('/content/drive')
OUT_DIR = '/content/drive/MyDrive/pubmed_embedder'
import os; os.makedirs(OUT_DIR, exist_ok=True)

MessageError: User cancelled dfs_ephemeral authorization

In [2]:
!mkdir pubmed && cd pubmed


In [4]:
from urllib.request import urlretrieve

for i in ["0001", "0002", "0003", "0004", "0005", "0006", "0007", "0008"]:
    urlretrieve(f"https://ftp.ncbi.nlm.nih.gov/pubmed/baseline/pubmed26n{i}.xml.gz", f"pubmed/pubmed26n{i}.xml.gz")

In [5]:
dataset_dir = "pubmed"

from os import listdir
from os.path import isfile, join
datasets_file_names = [f for f in listdir(dataset_dir) if isfile(join(dataset_dir, f))]

In [ ]:


def text_of(e):
    return ' '.join(''.join(e.itertext()).split()) if e is not None else ''

def extract_articles(file_path):
    with gzip.open(file_path, 'rb') as f:
        for _, elem in ET.iterparse(f, events=("end",)):
            if elem.tag != "PubmedArticle":
                continue
            art = elem.find("MedlineCitation/Article")
            if art is not None:
                title = text_of(art.find("ArticleTitle"))
                parts = [text_of(p) for p in art.findall("Abstract/AbstractText")]
                abstract = ' '.join(p for p in parts if p)
                if title and abstract:
                    yield title, abstract
            elem.clear()


In [ ]:
pairs, seen = [], set()
for path in sorted(glob.glob('pubmed/*.xml.gz')):
    for t, a in extract_articles(path):
        if len(a.split()) < MIN_ABS_WORDS or len(t.split()) < 4 or t in seen:
            continue
        seen.add(t); pairs.append((t, a))
    print(path, len(pairs))

random.shuffle(pairs)
pairs = pairs[:MAX_PAIRS]
print(len(pairs), 'pairs')

pubmed/pubmed26n0001.xml.gz 14922
pubmed/pubmed26n0002.xml.gz 27911
pubmed/pubmed26n0003.xml.gz 40043
pubmed/pubmed26n0004.xml.gz 54382
pubmed/pubmed26n0005.xml.gz 67728
pubmed/pubmed26n0006.xml.gz 83431
pubmed/pubmed26n0007.xml.gz 99234
pubmed/pubmed26n0008.xml.gz 114132
114132 pairs; 300000 sentences


In [79]:
import nltk
nltk.download('punkt', quiet=True); nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import sent_tokenize

dev = pairs[:2000]              
train_pairs = pairs[2000:]

MAX_SENTENCES = 300_000
sentences = [s for _, a in train_pairs for s in sent_tokenize(a) if len(s.split()) >= 6]
random.shuffle(sentences)
sentences = sentences[:MAX_SENTENCES]
print(len(dev), 'dev pairs;', len(sentences), 'TSDAE sentences')

2000 dev pairs; 300000 TSDAE sentences


In [ ]:
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, models, datasets, losses

BASE_MODEL = 'bert-base-uncased'
TSDAE_SENTENCES = 20_000      # use 300_000 for the real run
TSDAE_BATCH = 16

word = models.Transformer(BASE_MODEL, max_seq_length=128)
pool = models.Pooling(word.get_word_embedding_dimension(), 'cls')
model = SentenceTransformer(modules=[word, pool])

ds = datasets.DenoisingAutoEncoderDataset(sentences[:TSDAE_SENTENCES])
loader = DataLoader(ds, batch_size=TSDAE_BATCH, shuffle=True, drop_last=True)
loss = losses.DenoisingAutoEncoderLoss(model, decoder_name_or_path=BASE_MODEL,
                                       tie_encoder_decoder=True)

model.fit(train_objectives=[(loader, loss)], epochs=1, weight_decay=0,
          scheduler='constantlr', optimizer_params={'lr': 3e-5},
          show_progress_bar=True)
model.save('tsdae-domain-model')
!cp -r tsdae-domain-model "{OUT_DIR}/"

/tmp/ipykernel_2349/417674504.py:2: DeprecationWarning: Importing from 'sentence_transformers.models' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.modules' instead.
  from sentence_transformers import SentenceTransformer, models, datasets, losses
/tmp/ipykernel_2349/417674504.py:2: DeprecationWarning: Importing from 'sentence_transformers.datasets' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.datasets' instead.
  from sentence_transformers import SentenceTransformer, models, datasets, losses
/tmp/ipykernel_2349/417674504.py:2: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, models, datasets, losses
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertLMHeadModel LOAD REPORT from: bert-base-uncased
Key                                                                | Status     | 
-------------------------------------------------------------------+------------+-
bert.pooler.dense.bias                                             | UNEXPECTED | 
cls.seq_relationship.weight                                        | UNEXPECTED | 
bert.pooler.dense.weight                                           | UNEXPECTED | 
cls.seq_relationship.bias                                          | UNEXPECTED | 
bert.encoder.layer.{0...11}.crossattention.self.query.weight       | MISSING    | 
bert.encoder.layer.{0...11}.crossattention.self.key.bias           | MISSING    | 
bert.encoder.layer.{0...11}.crossattention.output.LayerNorm.bias   | MISSING    | 
bert.encoder.layer.{0...11}.crossattention.self.value.weight       | MISSING    | 
bert.encoder.layer.{0...11}.crossattention.self.query.bias         | MISSING    | 
bert.encoder.layer.{

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
500,4.842410
1000,4.049747


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## 2- SBERT

In [76]:
from datasets import load_dataset
from sentence_transformers import InputExample
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator, SimilarityFunction

nli = load_dataset("sentence-transformers/all-nli", "triplet", split="train")
nli = nli.shuffle(seed=42).select(range(100_000))
nli_examples = [InputExample(texts=[r["anchor"], r["positive"], r["negative"]])
                for r in nli]

stsb = load_dataset("sentence-transformers/stsb", split="validation")
stsb_eval = EmbeddingSimilarityEvaluator(
    stsb["sentence1"], stsb["sentence2"], stsb["score"],
    main_similarity=SimilarityFunction.COSINE, name="stsb-dev")
print(len(nli_examples), 'NLI triplets')

/tmp/ipykernel_2349/840684073.py:3: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator, SimilarityFunction


README.md:   0%|          | 0.00/5.15k [00:00<?, ?B/s]

triplet/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 38.4MB            

triplet/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

triplet/dev-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  782kB            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/5749 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1379 [00:00<?, ? examples/s]

100000 NLI triplets


In [77]:
def train_on_nli(start, out):
    if start is None:
        m = SentenceTransformer(modules=[
            models.Transformer('bert-base-uncased', max_seq_length=128),
            models.Pooling(768, 'cls')])
    else:
        m = SentenceTransformer(start)
        m.max_seq_length = 128
    loader = DataLoader(nli_examples, shuffle=True, batch_size=32, drop_last=True)
    loss = losses.MultipleNegativesRankingLoss(m)
    m.fit(train_objectives=[(loader, loss)], evaluator=stsb_eval, epochs=1,
          warmup_steps=int(len(loader) * 0.1), optimizer_params={'lr': 2e-5},
          output_path=out, save_best_model=True, show_progress_bar=True)
    return SentenceTransformer(out)

import os
TSDAE_PATH = '/content/tsdae-domain-model'
assert os.path.isdir(TSDAE_PATH), "TSDAE folder not found - rerun Cell 6 or copy it back from Drive"

nli_only  = train_on_nli(None, 'bert-nli-only')
tsdae_nli = train_on_nli(TSDAE_PATH, 'tsdae-nli')
!cp -r bert-nli-only tsdae-nli "{OUT_DIR}/"

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## Evaluator

In [80]:
import pandas as pd
from sentence_transformers.evaluation import InformationRetrievalEvaluator, NanoBEIREvaluator

def make_evaluator(p, name):
    q = {f'q{i}': a for i, (a, _) in enumerate(p)}
    c = {f'd{i}': b for i, (_, b) in enumerate(p)}
    r = {f'q{i}': {f'd{i}'} for i in range(len(p))}
    return InformationRetrievalEvaluator(q, c, r, name=name, mrr_at_k=[10],
                                         ndcg_at_k=[10], accuracy_at_k=[1, 5, 10])

pubmed_eval = make_evaluator(dev, 'pubmed-dev')
nano = NanoBEIREvaluator(dataset_names=["scifact", "nfcorpus", "scidocs"])

plain = SentenceTransformer(modules=[
    models.Transformer('bert-base-uncased', max_seq_length=256),
    models.Pooling(768, 'cls')])
tsdae_only = SentenceTransformer('tsdae-domain-model')

candidates = {
    'A. plain bert': plain,
    'B. TSDAE only': tsdae_only,
    'C. bert + NLI (no TSDAE)': nli_only,
    'D. TSDAE + NLI (your method)': tsdae_nli,
}

def pick(res, key):
    return next(v for k, v in res.items() if key in k)

rows = {}
for name, m in candidates.items():
    m.max_seq_length = 256
    rows[name] = {
        'PubMed title->abstract nDCG@10': pick(pubmed_eval(m), 'ndcg@10'),
        'NanoBEIR bio mean nDCG@10': pick(nano(m), 'mean_cosine_ndcg@10'),
        'STSb dev Spearman': pick(stsb_eval(m), 'spearman_cosine'),
    }
    print(name, 'done')

results = pd.DataFrame(rows).T.round(4)
display(results)
results.to_csv(f'{OUT_DIR}/results.csv')

Loading NanoBEIR datasets:   0%|          | 0/3 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/7.15k [00:00<?, ?B/s]

corpus/NanoClimateFEVER-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 3.29MB            

corpus/NanoClimateFEVER-00000-of-00001.p(…): downloading bytes:           |  0.00B            

corpus/NanoDBPedia-00000-of-00001.parque(…): reconstructing file:   0%|          |  0.00B / 1.46MB            

...

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

A. plain bert done
B. TSDAE only done
C. bert + NLI (no TSDAE) done
D. TSDAE + NLI (your method) done


,PubMed title->abstract nDCG@10,NanoBEIR bio mean nDCG@10,STSb dev Spearman
A. plain bert,0.0420,0.0162,0.3173
B. TSDAE only,0.2881,0.0799,0.6431
C. bert + NLI (no TSDAE),0.4939,0.1825,0.8355
D. TSDAE + NLI (your method),0.4926,0.1737,0.8350


NameError: name 'OUT_DIR' is not defined